<a href="https://colab.research.google.com/github/justinh613/BA305-A1-TeamA2/blob/main/Recipe_Popularity_Prediction_Ridge_%2B_Lasso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Predictive Question:** Can you predict the rating of a recipe based on its ingredients?

**Method:** Ridge + Lasso Regression

In [ ]:
# IMPORT PACKAGES

# Data handling
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Splitting and preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MultiLabelBinarizer
import re
from collections import Counter


# Models
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, ElasticNet

# Evaluation (regression)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# IMPORT DATASET

url = "https://raw.githubusercontent.com/jalmendral/BA305-A1-TeamA2/main/Data/processed/recipes_clean.csv"
df = pd.read_csv(url)

In [ ]:
# KENNY'S ADDITIONAL CLEANING

# Keep recipes with both ingredient text and an observed rating.
model_df = df.dropna(subset=["ingredients", "avg_rating"]).copy()

# X is the information used to predict.
# y is the value we want to predict.
X = model_df["ingredients"]
y = model_df["avg_rating"]

In [ ]:
# KENNY'S TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [ ]:
# STRING CLEANING + DUMMY CREATION

import re
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Compilation of constants and regular expressions
FRACS = "¼½¾⅐⅑⅒⅓⅔⅕⅖⅗⅘⅙⅚⅛⅜⅝⅞"
SPLIT = re.compile(rf",\s*(?=[\d{FRACS}])")      # commas that start a new ingredient
QTY   = re.compile(rf"^[\d\s/.\-{FRACS}]+")      # leading numbers/fractions
UNITS = re.compile(
    r"^(?:cups?|tablespoons?|teaspoons?|tbsp|tsp|ounces?|oz|pounds?|lbs?|"
    r"pinch(?:es)?|dash(?:es)?|cans?|packages?|pkg|cloves?|sticks?|slices?|"
    r"pints?|quarts?|gallons?)\b\.?\s*(?:of\s+)?"
)
DESCRIPTORS = re.compile(
    r"\b(?:packed|softened|melted|chopped|diced|minced|sliced|grated|shredded|"
    r"large|small|medium|fresh|freshly|ground|divided|beaten|sifted|toasted)\b"
)

def clean_ingredient_item(item):
    if not isinstance(item, str):
        return ""

    # Basic cleaning (lowercasing, parentheticals, post-comma descriptions)
    item = item.lower()
    item = re.sub(r"\(.*?\)", "", item)      # drop parentheticals like "(8 ounce)"
    item = item.split(",")[0]                # drop prep notes after a comma

    # Remove measurements and units
    item = QTY.sub("", item).strip()
    item = UNITS.sub("", item)
    item = DESCRIPTORS.sub("", item)

    # Refined Cleaning: Remove leading symbols (hyphens, percents, bullet points, etc.), 'or', and leftover digits
    item = re.sub(r"^(?:%|or|\d%|\d|\s|[-#*+])+", "", item).strip()

    # Normalize white spaces
    item = re.sub(r"\s+", " ", item).strip()

    # Discard if too short or if it contains only symbols/numbers
    if len(item) < 2 or re.match(r"^[\d\W_]+$", item):
        return ""

    return item

def parse_ingredients_comprehensive(s):
    if not isinstance(s, str):
        return []
    items = [clean_ingredient_item(x) for x in SPLIT.split(s)]
    return sorted({i for i in items if i})   # unique, non-empty

# Apply the combined cleaning pipeline to datasets
df["ing_list"] = df["ingredients"].apply(parse_ingredients_comprehensive)
model_df["ing_list"] = model_df["ingredients"].apply(parse_ingredients_comprehensive)

# Generate dummy variables
mlb_comprehensive = MultiLabelBinarizer()
ingredients_encoded_comp = mlb_comprehensive.fit_transform(model_df['ing_list'])
raw_ingredients_df = pd.DataFrame(ingredients_encoded_comp, columns=mlb_comprehensive.classes_, index=model_df.index)

# Filter out columns starting with leading symbols or non-word characters for safety
clean_cols = [col for col in raw_ingredients_df.columns if not re.match(r'^[\W_]', col)]
ingredients_df = raw_ingredients_df[clean_cols]

In [ ]:
# BASELINE MEAN MODEL

# Calculate the mean rating from the training set
mean_rating_train = y_train.mean()

# Create predictions consisting entirely of this mean value
y_pred_baseline_train = np.full_like(y_train, fill_value=mean_rating_train)
y_pred_baseline_test = np.full_like(y_test, fill_value=mean_rating_train)

# Calculate RMSE for the baseline predictions
baseline_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_baseline_train))
baseline_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline_test))

print(f"Baseline Mean Prediction: {mean_rating_train:.4f}")
print(f"Baseline Train RMSE: {baseline_train_rmse:.4f}")
print(f"Baseline Test RMSE: {baseline_test_rmse:.4f}")

Baseline Mean Prediction: 4.5217
Baseline Train RMSE: 0.4107
Baseline Test RMSE: 0.4147


In [ ]:
# RIDGE REGRESSION

# Align our cleaned dummy variables with the train/test split indices
X_train_ingredients = ingredients_df.loc[X_train.index]
X_test_ingredients = ingredients_df.loc[X_test.index]

# Define a reasonable range of alphas (regularization strengths)
ridge_alphas = np.logspace(-2, 3, 20)

# Set up Ridge with 5-Fold Cross Validation
ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5, scoring='neg_mean_squared_error')

# Fit the cross-validated Ridge model on the training data
print("Training Ridge regression with 5-fold cross-validation...")
ridge_cv.fit(X_train_ingredients, y_train)

# Evaluate the optimal model on train and test sets
y_pred_ridge_train = ridge_cv.predict(X_train_ingredients)
y_pred_ridge_test = ridge_cv.predict(X_test_ingredients)

ridge_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_ridge_train))
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge_test))
ridge_test_r2 = r2_score(y_test, y_pred_ridge_test)

print(f"\nOptimal Alpha (Lambda): {ridge_cv.alpha_:.6f}")
print(f"Train RMSE: {ridge_train_rmse:.4f}")
print(f"Test RMSE: {ridge_test_rmse:.4f}")
print(f"Test R2 Score: {ridge_test_r2:.4f}")

Training Ridge regression with 5-fold cross-validation...

Optimal Alpha (Lambda): 48.329302
Train RMSE: 0.3805
Test RMSE: 0.4101
Test R2 Score: 0.0220


In [ ]:
# LASSO REGRESSION

# Align our cleaned dummy variables with the train/test split indices
X_train_ingredients = ingredients_df.loc[X_train.index]
X_test_ingredients = ingredients_df.loc[X_test.index]

# Define a reasonable range of alphas (regularization strengths, often called lambda)
alphas = np.logspace(-4, 1, 20)

# Set up Lasso with 5-Fold Cross Validation
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42, n_jobs=-1, max_iter=2000)

# Fit the cross-validated Lasso model on the training data
print("Training Lasso regression with 5-fold cross-validation...")
lasso_cv.fit(X_train_ingredients, y_train)

# Evaluate the optimal model on train and test sets
y_pred_train = lasso_cv.predict(X_train_ingredients)
y_pred_test = lasso_cv.predict(X_test_ingredients)

train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

print(f"\nOptimal Alpha (Lambda): {lasso_cv.alpha_:.6f}")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R2 Score: {test_r2:.4f}")

# Show non-zero coefficients
coef_series = pd.Series(lasso_cv.coef_, index=ingredients_df.columns)
non_zero_coefs = coef_series[coef_series != 0]
print(f"\nNumber of non-zero features selected: {len(non_zero_coefs)} out of {ingredients_df.shape[1]}")

Training Lasso regression with 5-fold cross-validation...

Optimal Alpha (Lambda): 0.000336
Train RMSE: 0.3981
Test RMSE: 0.4108
Test R2 Score: 0.0186

Number of non-zero features selected: 200 out of 11403


**First Round Analysis Overview:**

After conducting ridge and lasso regression, I found that they were both time-consuming methods that did not reduce RMSE by any significant amount. This was likely due to string cleaning being insufficient, as there were still ~11,000 predictors, a common problem with trying to predict by strings. This prompts me to want to try and aggregate some of these strings into more useful groups.

In [ ]:
# CONSOLIDATED DYNAMIC INGREDIENT GROUPING & FREQUENCY FILTERING
import re
from collections import Counter
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Dynamically discover common head words (the last word of each unique ingredient)
all_unique_ingredients = list(ingredients_df.columns)
head_words = [ing.strip().split()[-1] for ing in all_unique_ingredients if ing.strip().split()]

# Find head words appearing at least 5 times in the unique ingredient vocabulary
head_word_counts = Counter(head_words)
DYNAMIC_LIMIT = 5
common_head_words = [word for word, count in head_word_counts.items() if count >= DYNAMIC_LIMIT and len(word) > 2]

# Helper to map complex ingredients to our discovered base categories
def dynamic_group_ingredient(item):
    for keyword in common_head_words:
        if f" {keyword}" in f" {item}":
            return keyword
    return item

# 2. Apply the dynamic grouping map to the ingredient lists
model_df["dynamic_grouped_ing_list"] = model_df["ing_list"].apply(
    lambda ing_list: sorted({dynamic_group_ingredient(ing) for ing in ing_list})
)

# 3. Create dummy variables from the grouped ingredients
mlb_dynamic = MultiLabelBinarizer()
ingredients_encoded_dynamic = mlb_dynamic.fit_transform(model_df['dynamic_grouped_ing_list'])
raw_ingredients_dynamic_df = pd.DataFrame(
    ingredients_encoded_dynamic,
    columns=mlb_dynamic.classes_,
    index=model_df.index
)

# Remove any remaining columns that begin with non-word characters for safety
clean_dynamic_cols = [col for col in raw_ingredients_dynamic_df.columns if not re.match(r'^[^\w]', col)]
dynamic_ingredients_df = raw_ingredients_dynamic_df[clean_dynamic_cols]

# 4. Filter out any ingredient appearing in fewer than 10 recipes total
dynamic_frequencies = dynamic_ingredients_df.sum(axis=0)
filtered_dynamic_ingredients_df = dynamic_ingredients_df[dynamic_frequencies[dynamic_frequencies >= 10].index]

# 5. Output summary
print("=== CONSOLIDATION SUMMARY ===")
print(f"Original vocabulary size: {ingredients_df.shape[1]} columns")
print(f"Dynamically discovered base categories: {len(common_head_words)}")
print(f"Final simplified features (Grouped & appearing >= 10 times): {filtered_dynamic_ingredients_df.shape[1]} columns")
print("\nTop 15 most frequent consolidated ingredients:")
display(dynamic_frequencies.loc[filtered_dynamic_ingredients_df.columns].sort_values(ascending=False).head(15))

=== CONSOLIDATION SUMMARY ===
Original vocabulary size: 11403 columns
Dynamically discovered base categories: 402
Final simplified features (Grouped & appearing >= 10 times): 369 columns

Top 15 most frequent consolidated ingredients:


,0
salt,6409
sugar,5783
butter,5111
oil,5045
onion,4874
pepper,4596
flour,4434
garlic,3733
cheese,3709
powder,3473


In [ ]:
# RIDGE REGRESSION WITH CONSOLIDATED INGREDIENTS

# Align our simplified/consolidated dummy variables with the train/test split indices
X_train_consolidated = filtered_dynamic_ingredients_df.loc[X_train.index]
X_test_consolidated = filtered_dynamic_ingredients_df.loc[X_test.index]

# Define a reasonable range of alphas (regularization strengths)
ridge_alphas = np.logspace(-2, 3, 20)

# Set up Ridge with 5-Fold Cross Validation
ridge_cv_cons = RidgeCV(alphas=ridge_alphas, cv=5, scoring='neg_mean_squared_error')

# Fit the cross-validated Ridge model on the consolidated training data
print("Training Ridge regression with 5-fold cross-validation using consolidated ingredients...")
ridge_cv_cons.fit(X_train_consolidated, y_train)

# Evaluate the optimal model on train and test sets
y_pred_ridge_train_cons = ridge_cv_cons.predict(X_train_consolidated)
y_pred_ridge_test_cons = ridge_cv_cons.predict(X_test_consolidated)

ridge_train_rmse_cons = np.sqrt(mean_squared_error(y_train, y_pred_ridge_train_cons))
ridge_test_rmse_cons = np.sqrt(mean_squared_error(y_test, y_pred_ridge_test_cons))
ridge_test_r2_cons = r2_score(y_test, y_pred_ridge_test_cons)

print(f"\nOptimal Alpha (Lambda): {ridge_cv_cons.alpha_:.6f}")
print(f"Consolidated Train RMSE: {ridge_train_rmse_cons:.4f}")
print(f"Consolidated Test RMSE: {ridge_test_rmse_cons:.4f}")
print(f"Consolidated Test R2 Score: {ridge_test_r2_cons:.4f}")

Training Ridge regression with 5-fold cross-validation using consolidated ingredients...

Optimal Alpha (Lambda): 48.329302
Consolidated Train RMSE: 0.3982
Consolidated Test RMSE: 0.4112
Consolidated Test R2 Score: 0.0167


In [ ]:
# LASSO REGRESSION WITH CONSOLIDATED INGREDIENTS

# Align our simplified/consolidated dummy variables with the train/test split indices
X_train_consolidated = filtered_dynamic_ingredients_df.loc[X_train.index]
X_test_consolidated = filtered_dynamic_ingredients_df.loc[X_test.index]

# Define a reasonable range of alphas (regularization strengths)
lasso_alphas = np.logspace(-4, 1, 20)

# Set up Lasso with 5-Fold Cross Validation
lasso_cv_cons = LassoCV(alphas=lasso_alphas, cv=5, random_state=42, n_jobs=-1, max_iter=2000)

# Fit the cross-validated Lasso model on the consolidated training data
print("Training Lasso regression with 5-fold cross-validation using consolidated ingredients...")
lasso_cv_cons.fit(X_train_consolidated, y_train)

# Evaluate the optimal model on train and test sets
y_pred_lasso_train_cons = lasso_cv_cons.predict(X_train_consolidated)
y_pred_lasso_test_cons = lasso_cv_cons.predict(X_test_consolidated)

lasso_train_rmse_cons = np.sqrt(mean_squared_error(y_train, y_pred_lasso_train_cons))
lasso_test_rmse_cons = np.sqrt(mean_squared_error(y_test, y_pred_lasso_test_cons))
lasso_test_r2_cons = r2_score(y_test, y_pred_lasso_test_cons)

print(f"\nOptimal Alpha (Lambda): {lasso_cv_cons.alpha_:.6f}")
print(f"Consolidated Lasso Train RMSE: {lasso_train_rmse_cons:.4f}")
print(f"Consolidated Lasso Test RMSE: {lasso_test_rmse_cons:.4f}")
print(f"Consolidated Lasso Test R2 Score: {lasso_test_r2_cons:.4f}")

# Show non-zero coefficients selected from the consolidated ingredients
coef_series_cons = pd.Series(lasso_cv_cons.coef_, index=filtered_dynamic_ingredients_df.columns)
non_zero_coefs_cons = coef_series_cons[coef_series_cons != 0]
print(f"\nNumber of non-zero features selected: {len(non_zero_coefs_cons)} out of {filtered_dynamic_ingredients_df.shape[1]}")

Training Lasso regression with 5-fold cross-validation using consolidated ingredients...

Optimal Alpha (Lambda): 0.000183
Consolidated Lasso Train RMSE: 0.3979
Consolidated Lasso Test RMSE: 0.4114
Consolidated Lasso Test R2 Score: 0.0156

Number of non-zero features selected: 205 out of 369


**Second Round Analysis Overview:**

After cutting down the 11,000 predictors into 402 base categories, I was able to almost completely recreate the predictive power of the first round of ridge and lasso regression. This tells me that much of the pattern in the data is noise and can be consolidated into a smaller number of parameters.